In [33]:
import pandas as pd
import numpy as np
from collections import Counter

In [34]:
data = pd.read_csv('/Users/romalarkin/Documents/Projects/LE/Cleaned_Life_Expectancy.csv')

In [ ]:
def fix_population_country(df_country, threshold=5):
    """
    I correct population values for a single country by enforcing
    temporal consistency and fixing scale-related reporting errors.

    The function assumes that sudden changes by an order of magnitude
    (×10 or ÷10) are more likely caused by inconsistent units
    (e.g. thousands vs millions) rather than real demographic changes.
    """
    df_country = df_country.sort_values("Year").copy() # I sort data chronologically to ensure year-to-year comparison
    pop = df_country["Population"].values.astype(float) # I extract population values as float for numerical operations

    fixed = pop.copy() # I initialize the corrected population array
    scale = 1.0 # I use a scale factor that is dynamically adjusted over time

    # I iterate through years starting from the second observation
    for i in range(1, len(pop)):
        prev = fixed[i - 1]
        curr = pop[i] * scale

        # If population drops unrealistically fast,
        # I assume a missing multiplier and scale the value up
        while curr < prev / threshold:
            scale *= 10
            curr = pop[i] * scale

        # If population increases unrealistically fast,
        # I assume an extra multiplier and scale the value down
        while curr > prev * threshold:
            scale /= 10
            curr = pop[i] * scale

        # I store the corrected value
        fixed[i] = curr

    # I overwrite the Population column with corrected values
    df_country["Population"] = fixed
    return df_country

In [ ]:
def fix_gdp_country(df_country, threshold_global=10, threshold_local=4):
    """
    I correct GDP per capita values for a single country by combining
    a global scale alignment with a local year-to-year consistency check.

    This function is designed for indicators where the absolute scale
    (hundreds vs thousands of USD) is critical.
    """
    df_country = df_country.sort_values("Year").copy()
    gdp = df_country["GDP"].values.astype(float)

    fixed = gdp.copy()

    # I use the maximum GDP value as a global reference scale
    ref = np.nanmax(gdp)

    for i in range(len(gdp)):
        curr = fixed[i]

        # I skip invalid or missing values
        if curr <= 0 or np.isnan(curr):
            continue

        # A. Global scale alignment
        # I bring all values to the same order of magnitude
        # as the largest observed GDP for this country
        while curr < ref / threshold_global:
            curr *= 10

        while curr > ref * threshold_global:
            curr /= 10

        # B. Local temporal consistency
        # I additionally protect against sharp year-to-year drops
        if i > 0:
            prev = fixed[i - 1]

            while curr < prev / threshold_local:
                curr *= 10

        fixed[i] = curr

    df_country["GDP"] = fixed
    return df_country

In [ ]:
def fix_metric_country(df_country, column, threshold=5):
    """
    I correct scale inconsistencies for a selected metric within a single country
    by enforcing year-to-year continuity.

    This function is intended for indicators where values should not
    change by an order of magnitude between consecutive years.
    """
    df_country = df_country.sort_values("Year").copy()
    values = df_country[column].values.astype(float)

    fixed = values.copy()

    for i in range(1, len(values)):
        prev = fixed[i - 1]
        curr = values[i]

        # I keep missing or zero values unchanged
        if curr <= 0 or np.isnan(curr):
            fixed[i] = curr
            continue

        # Sharp drop -> likely scale error (e.g. missing zero)
        while curr < prev / threshold:
            curr *= 10

        # Sharp spike -> inverse scale error
        while curr > prev * threshold:
            curr /= 10

        fixed[i] = curr

    df_country[column] = fixed
    return df_country


In [ ]:
def fix_column_country(df_country, column, threshold_drop=5, min_order_gap=1):
    """
    I correct scale inconsistencies for a given metric in a single country.
    This includes mode-based order-of-magnitude scaling and handling sharp drops.
    """

    # Sort by year to ensure chronological consistency
    df_country = df_country.sort_values("Year").copy()
    vals = df_country[column].astype(float).values
    fixed = vals.copy()

    # Replace zeros and negative values with NaN
    fixed[fixed <= 0] = np.nan

    # Work only if there are at least 3 valid numbers
    valid = fixed[~np.isnan(fixed)]
    if len(valid) < 3:
        return df_country

    # Determine the most common order of magnitude
    orders = np.floor(np.log10(valid)).astype(int)
    target_order = Counter(orders).most_common(1)[0][0]

    for i in range(len(fixed)):
        v = fixed[i]
        if np.isnan(v):
            continue

        curr_order = int(np.floor(np.log10(v)))

        # Mode-based scaling to match the most frequent order of magnitude
        if target_order - curr_order >= min_order_gap:
            v *= 10 ** (target_order - curr_order)
            curr_order = int(np.floor(np.log10(v)))

        # Sharp drop relative to previous year -> multiply by 10
        if i > 0:
            prev = fixed[i - 1]
            if not np.isnan(prev) and v < prev / threshold_drop:
                v *= 10

        # Update fixed value
        fixed[i] = v

    df_country[column] = fixed
    return df_country


In [ ]:
# Function to correct sudden drops in percentage-based vaccine data
def fix_vaccine_percentage_country(df_country, column, threshold_drop=5):
    """
    I correct sharp drops in vaccine coverage percentages for a single country.
    Values that fall abruptly compared to the previous year are assumed
    to be reporting artefacts and are scaled up.
    """
    # Sort data chronologically
    df_country = df_country.sort_values("Year").copy()
    vals = df_country[column].astype(float).values
    fixed = vals.copy()

    # Replace 0 values with NaN to avoid skewing the correction
    fixed[fixed == 0] = np.nan

    for i in range(1, len(fixed)):
        v = fixed[i]
        prev = fixed[i - 1]

        # Skip missing values
        if np.isnan(v) or np.isnan(prev):
            continue

        # Sharp drop compared to previous year -> multiply by 10
        if v < prev / threshold_drop:
            v *= 10

        # Cap values at 100% for logical consistency
        if v > 100:
            v = 100

        fixed[i] = v

    df_country[column] = fixed
    return df_country


In [ ]:
def fix_alcohol_country(df_country, threshold=5):
    """
    I correct sudden drops or spikes in alcohol consumption per capita
    for a single country. Sharp changes are assumed to be scale errors.
    """
    # Sort data chronologically
    df_country = df_country.sort_values("Year").copy()
    alc = df_country["Alcohol"].values.astype(float)

    fixed = alc.copy()
    scale = 1.0

    for i in range(1, len(alc)):
        prev = fixed[i - 1]
        curr = alc[i] * scale

        # Sharp drop -> multiply until within threshold
        while curr < prev / threshold:
            scale *= 10
            curr = alc[i] * scale

        # Sharp increase -> divide until within threshold
        while curr > prev * threshold:
            scale /= 10
            curr = alc[i] * scale

        fixed[i] = curr

    df_country["Alcohol"] = fixed
    return df_country

In [ ]:
def fix_thinness_country(df_country, column, threshold=5):
    """
    I correct sudden drops or spikes in thinness indicators for a single country.
    Sharp changes are assumed to be scale/reporting errors, and values are capped
    between 0% and 100%.
    """
    # Sort data chronologically
    df_country = df_country.sort_values("Year").copy()
    vals = df_country[column].astype(float).values

    fixed = vals.copy()
    scale = 1.0

    for i in range(1, len(vals)):
        prev = fixed[i - 1]
        curr = vals[i] * scale

        # Sharp drop -> multiply until within threshold
        while curr < prev / threshold:
            scale *= 10
            curr = vals[i] * scale

        # Sharp increase -> divide until within threshold
        while curr > prev * threshold:
            scale /= 10
            curr = vals[i] * scale

        # Clamp values to 0-100%
        if curr > 100:
            curr = 100
        if curr < 0:
            curr = 0

        fixed[i] = curr

    df_country[column] = fixed
    return df_country


In [ ]:
# --- Manual corrections ---
# This code block applies manual fixes in cases where the initial data
# is incorrect and automated functions are not able to correct it reliably.

#Population fixes
for country in ["China", "India"]:
    data.loc[data["Country"] == country, "Population"] *= 1000

for country in ["Belarus", "Mexico", "Turkey", "France", "Germany", "Italy", "Spain"]:
    data.loc[data["Country"] == country, "Population"] *= 10

#percentage expenditure fixes
for country in ["Ukraine"]:
    data.loc[data["Country"] == country, "percentage expenditure"] /= 10

In [43]:
data_fixed = (
    data
    .groupby("Country", group_keys=False)
    .apply(fix_population_country)
    .groupby("Country", group_keys=False)
    .apply(fix_gdp_country)
    .groupby("Country", group_keys=False)
    .apply(lambda x: fix_metric_country(x, "BMI", threshold=4))
    .groupby("Country", group_keys=False)
    .apply(lambda x: fix_metric_country(x, "Adult Mortality", threshold=4))
    .groupby("Country", group_keys=False)
    .apply(lambda x: fix_column_country(x, column="percentage expenditure"))
    .groupby("Country", group_keys=False)
    .apply(lambda x: fix_vaccine_percentage_country(x, column="Hepatitis B"))
    .groupby("Country", group_keys=False)
    .apply(lambda x: fix_vaccine_percentage_country(x, column="Diphtheria"))
    .groupby("Country", group_keys=False)
    .apply(lambda x: fix_vaccine_percentage_country(x, column="Polio"))
    .groupby("Country", group_keys=False)
    .apply(fix_alcohol_country)
    .groupby("Country", group_keys=False)
    .apply(lambda x: fix_thinness_country(x, column="thinness 1-19 years"))
    .groupby("Country", group_keys=False)
    .apply(lambda x: fix_thinness_country(x, column="thinness 5-9 years"))

)

/var/folders/62/lvhscghj6zg6hvx3m8z7l3bw0000gn/T/ipykernel_56510/777433425.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data
/var/folders/62/lvhscghj6zg6hvx3m8z7l3bw0000gn/T/ipykernel_56510/777433425.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data
/var/folders/62/lvhscghj6zg6hvx3m8z7l3bw0000gn/T/ipykernel_56510/777433425.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. Thi

In [44]:
data_fixed.to_csv(
    '/Users/romalarkin/Documents/Projects/LE/Cleaned_Life_Expectancy_FIXED.csv',
    index=False
)